# Atividade Prática — Aula 6: Visualização Interativa com Plotly

Esta atividade foi construída com base nos slides da Aula 6, que apresentam a transição do gráfico estático para o gráfico como **aplicativo de exploração**, com foco em **hover**, **zoom**, **pan**, **filtros visuais**, **Plotly Express** e construção de componentes que mais tarde podem virar um dashboard. fileciteturn7file0

## Ideia central da aula
A interatividade existe para apoiar a investigação do gestor em tempo real.  
Mesmo assim, a aula reforça uma regra essencial: **interatividade não substitui clareza**. O gráfico precisa continuar limpo, bem titulado e orientado à decisão. fileciteturn7file0

## Regras da atividade
- O notebook orienta, mas **você deve construir os códigos**.
- Use **Plotly Express** sempre que possível.
- Após cada visual principal, escreva uma breve **interpretação humana**.
- Teste hover, zoom e isolamento por legenda antes de concluir.
- Pense como desenvolvedor: os gráficos de hoje poderão virar o dashboard de amanhã. fileciteturn7file0

## Dataset da atividade
Arquivo: `vendas_brasil_clean_aula6_plotly.csv`


## 1. Preparação do ambiente

Importe as bibliotecas necessárias.

**Sugestão:**
- `pandas`
- `plotly.express`
- `plotly.graph_objects` (opcional)


In [2]:
import pandas as pd
import plotly.express as px

## 2. Leitura e inspeção inicial da base

Leia o arquivo `vendas_brasil_clean_aula6_plotly.csv` em um DataFrame chamado `df`.

Depois:
1. exiba as primeiras linhas
2. verifique o tamanho da base
3. confira os tipos das colunas
4. confirme se `data_venda` está em formato adequado para análises temporais
5. identifique quais colunas podem alimentar:
   - comparação de categorias
   - evolução no tempo
   - relação entre variáveis
   - distribuição espacial


In [4]:
df = pd.read_csv('vendas_brasil_clean_aula6_plotly.csv')

print('Primeiras 5 linhas do DataFrame:')
display(df.head())

print('\nTamanho da base (linhas, colunas):')
print(df.shape)

print('\nTipos de dados das colunas:')
df.info()

df['data_venda'] = pd.to_datetime(df['data_venda'], errors='coerce')
print('\nVerificação do tipo de data_venda após conversão:')
print(df['data_venda'].dtype)

print('\nColunas identificadas para análise:')
print('- Comparação de categorias: `canal_venda`, `categoria`, `produto`, `uf`')
print('- Evolução no tempo: `data_venda`')
print('- Relação entre variáveis: `receita`, `custo`, `lucro`, `quantidade`, `margem_lucro`')
print('- Distribuição espacial: `latitude`, `longitude`, `uf`')

Primeiras 5 linhas do DataFrame:


,pedido_id,data_venda,uf,canal_venda,categoria,produto,quantidade,desconto,preco_unitario,receita,lucro,margem_lucro,latitude,longitude,mes
0,9000,2025-01-29,RS,Marketplace,Acessórios,Teclado Mecânico,5,0.0700,386.96,4063.08,675.62,0.1663,-29.86937,-50.89149,2025-01
1,9001,2025-03-10,PE,App,Móveis,Mesa Compacta,3,0.1265,614.56,1843.68,271.35,0.1472,-8.58550,-35.20490,2025-03
2,9002,2025-03-25,SC,Online,Informática,Monitor 27,6,0.0712,1339.74,8038.44,2011.38,0.2502,-28.16059,-48.44543,2025-03
3,9003,2025-06-20,BA,Online,Móveis,Cadeira Office,4,0.0000,1004.15,4016.60,1326.00,0.3301,-13.82252,-38.83131,2025-06
4,9004,2025-07-19,RS,Online,Telefonia,Smartphone X,2,0.0000,2512.55,5025.10,762.04,0.1516,-29.44977,-51.64270,2025-07



Tamanho da base (linhas, colunas):
(560, 15)

Tipos de dados das colunas:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 560 entries, 0 to 559
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   pedido_id       560 non-null    int64  
 1   data_venda      560 non-null    object 
 2   uf              560 non-null    object 
 3   canal_venda     560 non-null    object 
 4   categoria       560 non-null    object 
 5   produto         560 non-null    object 
 6   quantidade      560 non-null    int64  
 7   desconto        560 non-null    float64
 8   preco_unitario  560 non-null    float64
 9   receita         560 non-null    float64
 10  lucro           560 non-null    float64
 11  margem_lucro    560 non-null    float64
 12  latitude        560 non-null    float64
 13  longitude       560 non-null    float64
 14  mes             560 non-null    object 
dtypes: float64(7), int64(2), object(6)
memory usage: 6

## 3. Matriz de decisão do analista

1.  **Por que esta atividade faz mais sentido em Plotly do que em Matplotlib?**
    Faz mais sentido em Plotly pela necessidade de **interatividade** (hover, zoom, filtros) para exploração dinâmica, algo que Matplotlib não oferece nativamente e exigiria muito mais esforço para replicar.

2.  **Em que situação você ainda preferiria um gráfico estático?**
    Preferiria um gráfico estático para **documentos impressos** (PDFs, relatórios) ou apresentações onde a exploração interativa não é necessária, ou para visualizações rápidas e simples onde a clareza estática é prioritária.

3.  **O destino desta análise parece mais “PDF” ou mais “tela/web”?**
    O destino desta análise é claramente **“tela/web”**, devido ao foco em interatividade, capacidade de responder a subperguntas e potencial para ser integrado em dashboards.

## 4. Missão 1 — Barras: Receita por Canal

A missão prática da aula pede construir um gráfico de barras para responder:
**qual canal vende mais?**  
Os slides também sugerem usar o hover para injetar métricas secundárias sem poluir a tela. fileciteturn7file0

### Tarefa
1. Agregue a `receita` por `canal_venda`
2. Inclua também uma métrica secundária, como `quantidade`
3. Construa um gráfico de barras com Plotly Express
4. Ordene do maior para o menor
5. Faça o hover mostrar mais do que apenas a receita

### Perguntas
- Qual canal lidera a receita?
- O canal líder também lidera em quantidade?
- O hover ajudou a enriquecer a leitura sem poluir a tela?


In [5]:
df_canal_receita = df.groupby('canal_venda')[['receita', 'quantidade']].sum().reset_index()
df_canal_receita = df_canal_receita.sort_values(by='receita', ascending=False)

fig = px.bar(
    df_canal_receita,
    x='canal_venda',
    y='receita',
    title='Receita Total por Canal de Venda',
    labels={'canal_venda': 'Canal de Venda', 'receita': 'Receita Total'},
    hover_data={'quantidade': True, 'receita': ':.2f'},
    color_discrete_sequence=px.colors.qualitative.Pastel
)

fig.update_layout(xaxis_title='Canal de Venda', yaxis_title='Receita Total')
fig.show()

### Insight obrigatório
Escreva 2 ou 3 linhas explicando:
- quem lidera
- quem fica atrás
- o que um gestor poderia investigar a seguir


### Insight obrigatório

O canal **Online** lidera com folga a receita total, seguido de perto pelo **App**. Em contraste, o canal **Loja Física** apresenta a menor receita.

Um gestor poderia investigar a **rentabilidade** de cada canal, além da receita. Por exemplo, o canal Online pode ter maior receita, mas um custo de aquisição de cliente (CAC) mais elevado, ou margens menores devido a descontos agressivos. Comparar a margem de lucro por canal pode revelar oportunidades de otimização ou realocação de recursos.

## 5. Missão 2 — Linhas: Sazonalidade da Receita Mensal

Os slides destacam que o gráfico de linhas, com zoom e pan, é ideal para navegar no tempo e isolar picos. Também reforçam que o eixo temporal precisa estar corretamente tipado no Pandas. fileciteturn7file0

### Tarefa
1. Converta `data_venda` em data, se necessário
2. Agregue a `receita` por mês
3. Construa um gráfico de linha interativo
4. Teste o zoom nos meses de pico
5. Observe se novembro e dezembro se destacam

### Perguntas
- Existe sazonalidade?
- Quais meses chamam atenção?
- O zoom ajudou a explorar melhor a parte final da série?


In [8]:
df_receita_mensal = df.groupby(df['data_venda'].dt.to_period('M'))['receita'].sum().reset_index()
df_receita_mensal['data_venda'] = df_receita_mensal['data_venda'].dt.to_timestamp() # Converte para timestamp para Plotly

fig = px.line(
    df_receita_mensal,
    x='data_venda',
    y='receita',
    title='Sazonalidade da Receita Mensal',
    labels={'data_venda': 'Mês', 'receita': 'Receita Total'},
    markers=True,
    hover_name='data_venda'
)

fig.update_xaxes(dtick='M1', tickformat='%b-%Y')
fig.update_layout(xaxis_title='Mês', yaxis_title='Receita Total')
fig.show()

### Insight obrigatório

Existe uma **clara sazonalidade** na receita mensal, com **picos acentuados em novembro e dezembro**. Essa tendência é provavelmente impulsionada por **eventos sazonais de vendas** como a Black Friday e as compras de Natal. Um gestor poderia usar essa informação para otimizar o planejamento de estoque, marketing e recursos para esses períodos.

## 6. Missão 3 — Dispersão: Lucro vs. Receita segmentado por Categoria

Os slides mostram o uso do gráfico de dispersão para revelar correlação entre KPIs e identificar anomalias, com grande apoio do hover. fileciteturn7file0

### Tarefa
1. Construa um scatter plot com:
   - eixo X = `receita`
   - eixo Y = `lucro`
2. Use `categoria` como cor
3. Inclua no hover:
   - `produto`
   - `canal_venda`
   - `uf`
   - `margem_lucro`
4. Tente identificar algum ponto fora da curva

### Perguntas
- Existe correlação visual entre receita e lucro?
- Há anomalias?
- O hover ajuda a transformar um ponto em um caso investigável?


In [9]:
fig = px.scatter(
    df,
    x='receita',
    y='lucro',
    color='categoria',
    title='Lucro vs. Receita por Categoria',
    labels={'receita': 'Receita', 'lucro': 'Lucro'},
    hover_data={'produto': True, 'canal_venda': True, 'uf': True, 'margem_lucro': ':.2%'}, # Formata margem_lucro como porcentagem
    height=600,
    template='plotly_white'
)

fig.update_traces(marker=dict(size=10, opacity=0.8, line=dict(width=1, color='DarkSlateGrey')))
fig.update_layout(xaxis_title='Receita', yaxis_title='Lucro')
fig.show()

### Insight obrigatório

Existe uma **correlação positiva clara** entre receita e lucro. Pontos fora do padrão (anomalias) podem ser observados como os **pontos mais distantes da tendência geral**, especialmente em categorias com alta receita, mas lucro proporcionalmente menor ou maior. Ação analítica: **investigar individualmente** os produtos/vendas que formam esses pontos anômalos para entender suas particularidades (ex: altos descontos, custos elevados).

## 7. Exploração espacial — Onde está concentrada a operação?

Os slides incluem mapas geográficos como resposta para perguntas espaciais como:
**onde está concentrada nossa operação?** fileciteturn7file0

### Tarefa
Usando `latitude` e `longitude`, crie um mapa interativo que ajude a visualizar a operação.

### Sugestões
- use tamanho ou cor para uma métrica relevante (como receita)
- teste o zoom do mapa
- observe se há concentração regional

### Perguntas
- Onde a operação parece mais concentrada?
- O mapa ajudou mais do que uma simples tabela por UF?


In [10]:
fig = px.scatter_mapbox(
    df,
    lat='latitude',
    lon='longitude',
    color='uf',
    size='receita',
    hover_name='produto',
    hover_data={'receita': True, 'lucro': True, 'canal_venda': True, 'uf': True},
    zoom=4,
    title='Concentração da Operação por UF (Receita e Lucro)',
    mapbox_style='carto-positron',
    height=600
)

fig.update_layout(margin={'r':0,'t':50,'l':0,'b':0})
fig.show()


## 8. A tríade da interatividade

### Exemplo: Lucro vs. Receita por Categoria

1.  **O que o hover acrescenta:** Permite ver detalhes específicos de cada ponto (ex: `produto`, `canal_venda`, `UF`, `margem_lucro`), transformando um ponto genérico em um **caso investigável** sem sobrecarregar o visual principal.

2.  **Como o zoom melhora a investigação:** Ajuda a focar em regiões específicas do gráfico, como um aglomerado de pontos ou anomalias, para observar padrões ou identificar pontos de dados mais de perto que seriam difíceis de discernir na visão geral. Por exemplo, para analisar pontos de baixa receita e lucro.

3.  **Como o clique na legenda pode ajudar a isolar séries ou categorias:** Permite filtrar visualmente as categorias. Clicar em uma categoria na legenda a esconde, e um duplo clique isola apenas aquela categoria, facilitando a **comparação individual** ou a análise de grupos específicos.

## 9. Plotly Express — Máximo impacto, mínimo código

A aula mostra o Plotly Express como uma API declarativa, perfeita para DataFrames do Pandas e com geração automática de HTML/JS. fileciteturn7file0

### Tarefa
Explique em markdown:
1. O que significa dizer que Plotly Express é “declarativo”?
2. Em que ele facilita a vida do analista?
3. Como isso se conecta com a ideia de futuro dashboard em Streamlit?


## 9. Plotly Express — Máximo impacto, mínimo código

1.  **O que significa dizer que Plotly Express é “declarativo”?**
    Significa que você descreve *o que* quer visualizar (ex: 'gráfico de barras com X e Y') em vez de *como* construir cada elemento gráfico. O Plotly Express se encarrega dos detalhes de implementação.

2.  **Em que ele facilita a vida do analista?**
    Simplifica enormemente a criação de gráficos interativos complexos com poucas linhas de código, integrando-se facilmente com DataFrames do Pandas e gerando HTML/JS automaticamente, acelerando a exploração e comunicação de insights.

3.  **Como isso se conecta com a ideia de futuro dashboard em Streamlit?**
    Gráficos Plotly Express são nativamente interativos e baseados em HTML/JS, tornando-os ideais para incorporação direta em aplicações web como o Streamlit. A padronização e o código limpo do Plotly Express facilitam a transição de um script de análise para um componente de dashboard interativo e profissional.

## 11. O gráfico não fala sozinho

### Gráfico 1: Receita Total por Canal de Venda (Barras)
-   **Insight principal:** O canal Online domina a receita, seguido de perto pelo App, enquanto a Loja Física contribui significativamente menos.
-   **Decisão gerencial:** Considerar otimizar a alocação de recursos de marketing e vendas para os canais Online e App, e talvez reavaliar a estratégia para a Loja Física ou buscar oportunidades de crescimento nela.
-   **Pergunta adicional do gestor:** Qual a margem de lucro de cada canal? A rentabilidade acompanha a receita, ou há canais com receita menor mas margem de lucro superior?

### Gráfico 2: Sazonalidade da Receita Mensal (Linhas)
-   **Insight principal:** Existe uma forte sazonalidade anual, com picos de receita notáveis em novembro e dezembro, provavelmente impulsionados por datas comerciais como Black Friday e Natal.
-   **Decisão gerencial:** Planejar campanhas de marketing, gestão de estoque e alocação de equipe com antecedência para maximizar as vendas nos meses de pico e considerar estratégias para suavizar as quedas nos meses de menor movimento.
-   **Pergunta adicional do gestor:** Qual é a variação de produtos mais vendidos e as categorias de maior receita durante os picos de novembro/dezembro em comparação com outros meses? Isso pode direcionar o sortimento de produtos em cada período.

## 12. Missão prática final — Mapeando e construindo

### 1. Gráfico de Barras — Receita por Canal
-   **Pergunta de negócio:** Qual canal de venda gera mais receita?
-   **Colunas necessárias:** `canal_venda`, `receita`, `quantidade`
-   **Interpretação humana final:** O canal **Online** lidera a receita total, seguido pelo **App**. A **Loja Física** possui a menor receita. Este padrão sugere focar em estratégias para os canais digitais, enquanto se investiga a rentabilidade de cada um para otimizar recursos.

### 2. Gráfico de Linhas — Sazonalidade da Receita Mensal
-   **Pergunta de negócio:** A receita apresenta sazonalidade ao longo do ano?
-   **Colunas necessárias:** `data_venda`, `receita`
-   **Interpretação humana final:** Há uma **clara sazonalidade** com picos em **novembro e dezembro**, provavelmente devido à Black Friday e Natal. O gestor pode planejar ações de marketing, estoque e equipe para maximizar esses períodos de alta demanda e estratégias para os meses de menor movimento.

### 3. Gráfico de Dispersão — Lucro vs. Receita segmentado por Categoria
-   **Pergunta de negócio:** Existe correlação entre lucro e receita? Há produtos/categorias com performance atípica?
-   **Colunas necessárias:** `receita`, `lucro`, `categoria`, `produto`, `canal_venda`, `uf`, `margem_lucro`
-   **Interpretação humana final:** Uma **correlação positiva** entre receita e lucro é evidente. Pontos anômalos (com alta receita, mas lucro desproporcional, ou vice-versa) podem ser investigados individualmente, usando o hover, para entender os fatores específicos que influenciam sua performance.

### 4. Mapa — Concentração Geográfica da Operação (Extra)
-   **Pergunta de negócio:** Onde estão concentradas as operações de venda e quais regiões geram mais receita?
-   **Colunas necessárias:** `latitude`, `longitude`, `uf`, `receita`, `lucro`, `produto`, `canal_venda`
-   **Interpretação humana final:** O mapa visualiza a **concentração regional** das vendas, destacando áreas com maior receita e lucro. Isso auxilia na identificação de mercados fortes e na alocação estratégica de recursos para expansão ou otimização em diferentes estados (UFs).

## 13. Mindset de desenvolvedor

Um dos slides diz que os gráficos de hoje são o dashboard de amanhã.  
Isso exige padronização desde já: nomes coerentes, lógica limpa e reaproveitamento futuro. fileciteturn7file0

### Tarefa
Responda em markdown:
1. Como você nomearia seus objetos `fig_...` de forma organizada?
2. Que partes do seu código poderiam ser reaproveitadas numa aplicação web?
3. O que vale a pena padronizar desde esta aula?


## 13. Mindset de desenvolvedor

Um dos slides diz que os gráficos de hoje são o dashboard de amanhã.  
Isso exige padronização desde já: nomes coerentes, lógica limpa e reaproveitamento futuro. fileciteturn7file0

### Tarefa
Responda em markdown:
1. Como você nomearia seus objetos `fig_...` de forma organizada?
2. Que partes do seu código poderiam ser reaproveitadas numa aplicação web?
3. O que vale a pena padronizar desde esta aula?

### Respostas
1.  **Como você nomearia seus objetos `fig_...` de forma organizada?**
    Eu nomearia os objetos `fig_...` de forma a refletir o tipo de gráfico e a métrica principal que ele exibe, seguindo um padrão como `fig_tipo_metrica_dimensao`. Por exemplo:
    *   `fig_bar_receita_canal` (para o gráfico de barras de receita por canal)
    *   `fig_line_receita_mensal` (para o gráfico de linhas de receita mensal)
    *   `fig_scatter_lucro_receita_categoria` (para o gráfico de dispersão de lucro vs. receita por categoria)
    *   `fig_map_receita_geografica` (para o mapa de receita geográfica)
    
    Isso torna o código mais legível e fácil de entender a função de cada figura rapidamente.

2.  **Que partes do seu código poderiam ser reaproveitadas numa aplicação web?**
    As seguintes partes do código poderiam ser reaproveitadas em uma aplicação web (como Streamlit ou Flask/Dash):
    *   **Funções de pré-processamento de dados:** Código para carregar o CSV, converter tipos de dados (ex: `data_venda` para datetime), e agregar dados (`groupby`, `sum`, `reset_index`).
    *   **Criação dos gráficos Plotly Express:** Os objetos `fig` gerados pelo `px` são interativos por natureza e podem ser exportados diretamente para componentes HTML/JS que aplicações web podem renderizar.
    *   **Lógica de filtragem:** Se forem adicionados filtros (ex: por data, categoria), essa lógica de Pandas que manipula o DataFrame antes da visualização é diretamente reutilizável.
    *   **Temas e layouts:** `fig.update_layout()` e `fig.update_xaxes()` que padronizam a aparência e os eixos podem ser encapsulados para manter a consistência visual no dashboard.

3.  **O que vale a pena padronizar desde esta aula?**
    Para garantir um futuro dashboard organizado e de fácil manutenção, vale a pena padronizar desde já:
    *   **Nomenclatura de variáveis:** Usar nomes claros e consistentes para DataFrames (`df`, `df_canal_receita`), figuras (`fig_...`), e colunas.
    *   **Estilo de código:** Seguir guias de estilo (como PEP 8) para formatação, comentários e organização do código.
    *   **Uso de Plotly Express:** Manter a consistência na forma como os gráficos são construídos com Plotly Express, incluindo `labels`, `hover_data`, `title` e `template`.
    *   **Estrutura do notebook:** Organizar as seções logicamente (preparação, leitura, análises, insights) facilita a navegação e a extração de componentes para reutilização.
    *   **Funções para operações comuns:** Criar funções para tarefas repetitivas, como a agregação de dados ou a aplicação de um estilo de layout padrão aos gráficos. Isso promove o princípio DRY (Don't Repeat Yourself).

## 14. Desafio extra (opcional)

Crie mais um gráfico interativo à sua escolha, desde que ele responda uma pergunta real. Exemplos:
- receita por UF (barras)
- preço unitário ao longo do tempo (linhas)
- desconto vs. quantidade (dispersão)
- receita por categoria (barras horizontais)

Mas atenção:
- use hover com intenção
- teste zoom se fizer sentido
- escreva interpretação final


In [ ]:
# Desafio extra opcional


## 15. Entrega esperada

Seu notebook deve demonstrar:
- uso correto de Plotly Express
- escolha adequada do gráfico para a pergunta
- exploração consciente de hover, zoom, pan e legenda
- compromisso com clareza
- interpretação textual consistente

### Mensagem principal da aula
A tecnologia explora.  
Mas só o analista transforma exploração em decisão. fileciteturn7file0
